In [1]:
import sys
import os
from pathlib import Path
# Thêm thư mục cha (rag-service) vào danh sách tìm kiếm của Python
notebook_dir = Path(os.getcwd())
rag_service_dir = str(notebook_dir.parent.resolve())
if rag_service_dir not in sys.path:
    sys.path.append(rag_service_dir)


import time
import re
import requests
import json
import chromadb
from typing import List, Dict, Any
from configs.setting import settings
from configs.GetConfig import config
from src.a_ingestion.a1_loader import SupabaseDataLoader
from src.b_indexing.b0_vector_db import ChromaVectorDatabase


In [3]:

print(config.embedding_model)

nvidia/llama-nemotron-embed-vl-1b-v2:free


In [4]:
SuperbaseDataLoader = SupabaseDataLoader()
products = SuperbaseDataLoader.load_products()
policies = SuperbaseDataLoader.load_policies()

[Loader] Đang tải dữ liệu sản phẩm từ Supabase...
  -> Đã tải batch 0 - 999 (1000 sản phẩm)
  -> Đã tải batch 1000 - 1324 (325 sản phẩm)
[Loader] Đã tải thành công tổng cộng 1325 sản phẩm.
[Loader] Đang tải dữ liệu chính sách từ Supabase...
[Loader] Đã tải thành công 3 chính sách từ database.


In [ ]:
# Khởi tạo kết nối ChromaDB
client = ChromaVectorDatabase()

# Tách biệt làm 2 Collection chuyên biệt
product_col = client.get_or_create_collection(name="products_collection")
policy_col = client.get_or_create_collection(name="policies_collection")

print(f"Tổng vector sản phẩm: {product_col.count()}")
print(f"Tổng vector chính sách: {policy_col.count()}")

Tổng vector sản phẩm: 1325
Tổng vector chính sách: 3


In [ ]:
class LLMService:
    def __init__(self, settings):
        self.settings = settings

        self.openrouter_key = settings.OPENROUTER_API_KEY
        self.gemini_key = settings.GEMINI_API_KEY
        self.groq_key = settings.GROQ_API_KEY


    

In [ ]:
class RAGService:
    def __init__(self, config, EmbeddingService, VectorDatabase, settings, api_service):
        self.config = config
        self.EmbeddingService = EmbeddingService
        self.VectorDatabase = VectorDatabase

        self.openrouter_key = settings.OPENROUTER_API_KEY
        self.gemini_key = settings.GEMINI_API_KEY
        self.groq_key = settings.GROQ_API_KEY

        self.model_name

In [ ]:
import requests
import json
from configs.setting import settings

# Lấy API Key tự động từ cấu hình hệ thống
api_key = settings.OPENROUTER_API_KEY

if not api_key or "your-openrouter" in api_key:
    raise ValueError("Vui lòng điền OPENROUTER_API_KEY thật vào file .env!")

# Tạo payload đơn giản (truyền chuỗi văn bản trực tiếp vào "input")
payload = {
    "model": "nvidia/llama-nemotron-embed-vl-1b-v2:free",
    "input": sample_chunk_text,
    "encoding_format": "float"
}

response = requests.post(
    url="https://openrouter.ai/api/v1/embeddings",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    },
    data=json.dumps(payload)
)

result = response.json()
print(json.dumps(result, indent=2))

# Kiểm tra kết quả
if "error" in result:
    print("Lỗi từ OpenRouter API:", result["error"])
else:
    embedding = result["data"][0]["embedding"]
    print(f"Nhúng thành công! Số chiều của Vector (Dimension): {len(embedding)}")
    print("5 giá trị đầu tiên của Vector:", embedding[:5])
